# Credit Card Fraud Detection EDA & Modeling
This notebook analyzes the `fraudTrain.csv` dataset, handles highly imbalanced data, and trains multiple classification models to optimize for high recall.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')


## 1. Data Loading and Exploration

In [ ]:
# Using 10% sample of training data within Notebook for faster EDA
df_train_full = pd.read_csv('../data/fraudTrain.csv')
df = df_train_full.sample(frac=0.1, random_state=42).copy()
df.head()


## 2. Exploratory Data Analysis
Let's check the class distributions and basic feature patterns.

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='is_fraud')
plt.title('Fraud Class Distribution')
plt.show()

print(f"Fraud Ratio: {df['is_fraud'].mean():.4f}")


In [ ]:
# Transaction amount by class
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='is_fraud', y='amt')
plt.title('Transaction Amount by Fraud Status')
plt.yscale('log')
plt.show()


In [ ]:
# Fraud by Category
plt.figure(figsize=(12, 6))
sns.countplot(data=df[df['is_fraud']==1], y='category', order=df[df['is_fraud']==1]['category'].value_counts().index)
plt.title('Fraudulent Transactions by Category')
plt.show()


## 3. Preprocessing & SMOTE
Using the `data_utils.py` logic to standard scale numerical features and balance classes via SMOTE.

In [ ]:
import sys
sys.path.append('../src')
from data_utils import get_preprocessor

from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score, recall_score, confusion_matrix

df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
df['trans_dayofweek'] = df['trans_date_trans_time'].dt.dayofweek
df['trans_month'] = df['trans_date_trans_time'].dt.month

df['dob'] = pd.to_datetime(df['dob'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

features_to_keep = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 
                    'trans_hour', 'trans_dayofweek', 'trans_month', 'age', 'category', 'gender', 'is_fraud']

df = df[features_to_keep].dropna()

X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training on:", X_train.shape)


## 4. Modeling & Evaluation
Training a base logistic regression model prioritizing recall.

In [ ]:
preprocessor, _, _ = get_preprocessor()

# We only SMOTE inside the pipeline to prevent data leakage during CV
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy=0.1)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


## 5. Hyperparameter Tuning
Refer to `src/train.py` for full tuning setup using `RandomizedSearchCV` on Random Forests and Decision Trees. For real-world deployments on 350MB+ datasets, moving processing to dedicated scripts rather than Jupyter is standard practice.